# 02 — Clinical Feature Engineering, Model Tuning & MLflow Experiment Tracking
**Course**: Machine Learning Operations (MLOps) AIMLCZG523 — Assignment 01

Trains and tunes Logistic Regression, Random Forest, and Gradient Boosting classifiers with custom cardiovascular interaction features and 5-fold Stratified Cross-Validation.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import pandas as pd
from src.features.build_features import ClinicalFeatureEngineer, build_preprocessing_pipeline
from src.models.train import train_and_evaluate_models

In [2]:
# 1. Preview engineered cardiovascular features
df = pd.read_csv('../data/processed/heart_disease_cleaned.csv')
engineer = ClinicalFeatureEngineer()
df_engineered = engineer.fit_transform(df.drop(columns=['target']))
df_engineered[['age', 'trestbps', 'chol', 'thalach', 'hr_reserve_ratio', 'rate_pressure_product', 'chol_age_ratio', 'st_depression_per_hr']].head()

,age,trestbps,chol,thalach,hr_reserve_ratio,rate_pressure_product,chol_age_ratio,st_depression_per_hr
0,63.0,145.0,233.0,150.0,0.955414,21.75,3.698413,1.533333
1,67.0,160.0,286.0,108.0,0.705882,17.28,4.268657,1.388889
2,67.0,120.0,229.0,129.0,0.843137,15.48,3.417910,2.015504
3,37.0,130.0,250.0,187.0,1.021858,24.31,6.756757,1.871658
4,41.0,130.0,204.0,172.0,0.960894,22.36,4.975610,0.813953


In [3]:
# 2. Execute or load multi-model CV tuning, MLflow logging, and model packaging
meta_path = Path('../models/model_metadata.json')
if meta_path.exists():
    summary = json.loads(meta_path.read_text(encoding='utf-8'))
else:
    summary = train_and_evaluate_models(
        processed_csv='../data/processed/heart_disease_cleaned.csv',
        artifact_dir='../models',
        mlflow_tracking_uri='../mlruns'
    )
print('Winning Model:', summary['model_name'])
pd.DataFrame(summary['all_models_comparison']).T[[
    'cv_roc_auc_mean', 'cv_roc_auc_std', 'test_roc_auc',
    'test_accuracy', 'test_precision', 'test_recall', 'test_f1'
]]

Winning Model: Logistic_Regression


,cv_roc_auc_mean,cv_roc_auc_std,test_roc_auc,test_accuracy,test_precision,test_recall,test_f1
Logistic_Regression,0.9011,0.0138,0.9697,0.8852,0.8387,0.9286,0.8814
Random_Forest,0.8977,0.0236,0.9654,0.918,0.8966,0.9286,0.9123
Gradient_Boosting,0.884,0.0272,0.9545,0.8689,0.8571,0.8571,0.8571
